# Approximate Multi-Qubit Toffoli

In [ ]:
from qualtran import Bloq, CompositeBloq, BloqBuilder, Signature, Register
from qualtran import QBit, QInt, QUInt, QAny
from qualtran.drawing import show_bloq, show_call_graph, show_counts_sigma
from typing import *
import numpy as np
import sympy
import cirq

## `MultiAndLogDepth`
A log-depth many-bit AND with the same public I/O as `MultiAnd`.

This bloq computes the conjunction of a control register into a right-sided
output qubit using a balanced tree of Toffoli gates. The public register
interface matches `MultiAnd`: the input control register is preserved, the
output is written to a right-sided `target` qubit, and the intermediate tree
values are exposed as a right-sided `junk` register of length `n_ctrls - 2`.

The control pattern is specified by `cvs`. An entry `cvs[i] = 1` denotes a
positive control on `ctrl[i]`, while `cvs[i] = 0` denotes a negative control.
Negative controls are implemented by conjugating the corresponding wire with
`X` before and after the tree computation.

This implementation differs from the upstream `MultiAnd` in the semantics of
the `junk` register: here, `junk` stores balanced-tree intermediate values
rather than the ladder-style prefix-AND values used by the reference
implementation.

#### Parameters
 - `cvs`: A tuple of control values. Each entry must be `0` or `1`. The number of controls is `len(cvs)`, and must be at least `3`. 

#### Registers
 - `ctrl`: An `n`-bit control register.
 - `junk [right]`: An `n - 2` qubit junk register storing intermediate tree values.
 - `target [right]`: The output bit.


In [ ]:
from qualtran.bloqs.mcmt import MultiAndLogDepth

### Example Instances

In [ ]:
multi_and_log_depth = MultiAndLogDepth(cvs=(1, 1, 1, 1))

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([multi_and_log_depth],
           ['`multi_and_log_depth`'])

### Call Graph

In [ ]:
from qualtran.resource_counting.generalizers import ignore_split_join
multi_and_log_depth_g, multi_and_log_depth_sigma = multi_and_log_depth.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(multi_and_log_depth_g)
show_counts_sigma(multi_and_log_depth_sigma)

## `ParityMask`
Prepare masked copies of an input register for sampled subset-parity computation.

This bloq takes an $(n - 1)$-qubit input register `x` and prepares `k` output
registers `x_0, ..., x_{k-1}` according to a collection of classical sample
strings `s[0], ..., s[k-1]`. For each `i` and `j`,
the output bit $[x_i]_j = x_j \cdot s[i]_j$.

Equivalently, `x_i` is obtained by copying into the `i`-th output register
exactly those positions of `x` selected by the `i`-th mask, leaving all other
positions unchanged. The parity of each output string `x_i` therefore equals
the parity of the subset of input bits selected by `s[i]`.

This bloq is intended as a sample-preparation subroutine in an approximate
multi-controlled AND / Toffoli construction based on classically sampled
subsets. The sampling itself is assumed to have already been performed, and
the sampled subsets are provided explicitly through `sample_strings`.

#### Parameters
 - `n`: The total number of qubits in the target approximate Toffoli construction. This bloq acts on registers of length `n - 1`.
 - `k`: The number of sampled subsets, equivalently the number of output registers.
 - `sample_strings`: A tuple of `k` classical bit strings `s_1, ..., s_k`, each of length `n - 1`. The entry `s[i]_j` equals `1` iff the `j`-th input bit is included in the `i`-th sampled subset. 

#### Registers
 - `x`: An `(n - 1)`-qubit input register.
 - `x_0 [right]`: An `(n - 1)`-qubit output register containing the masked copy for the first sampled subset.
 - `x_1 [right]`: An `(n - 1)`-qubit output register containing the masked copy for the second sampled subset.
 - `...`: 
 - `x_{k-1} [right]`: An `(n - 1)`-qubit output register containing the masked copy for the last sampled subset.


In [ ]:
from qualtran.bloqs.mcmt import ParityMask

### Example Instances

In [ ]:
sample_strings = (
    (1, 0, 1, 0),
    (0, 1, 1, 0),
    (1, 1, 0, 1),
)
parity_mask = ParityMask(n=5, k=3, sample_strings=sample_strings)

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([parity_mask],
           ['`parity_mask`'])

### Call Graph

In [ ]:
from qualtran.resource_counting.generalizers import ignore_split_join
parity_mask_g, parity_mask_sigma = parity_mask.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(parity_mask_g)
show_counts_sigma(parity_mask_sigma)

## `ApproxMultiToffoli`
Approximate multi-controlled Toffoli via sampled subset parities.

This bloq implements an approximate `n`-qubit Toffoli / AND construction based on
classically sampled subsets. Given an `(n - 1)`-qubit control register `ctrl`, it
first prepares `k` masked copies determined by the sampled subset indicators in
`sample_strings`, computes the parity of each masked copy, negates those parity
bits, and then computes their conjunction into the output qubit `target`.

The resulting output approximates the `(n - 1)`-bit AND of the control register.
Equivalently, it approximates an `n`-qubit Toffoli with `(n - 1)` controls and one
target, except that this bloq exposes a fresh right-sided output qubit `target`
rather than toggling an input target in place.

All intermediate data are exposed through the right-sided `junk` register. In the
current implementation, `junk` is packed in the order
`x_0, ..., x_{k-1}, OR_values, ancilla`, where `x_i` are the masked copies,
`OR_values` are the computed parity bits, and `ancilla` are the auxiliary qubits
used by the multi-control AND subroutine.

#### Parameters
 - `n`: The total number of qubits in the target Toffoli interpretation. The bloq acts on an `(n - 1)`-qubit control register.
 - `k`: The number of sampled subsets used in the approximation.
 - `sample_strings`: A tuple of `k` classical bit strings, each of length `n - 1`. The entry `sample_strings[i][j]` equals `1` iff the `j`-th control bit is included in the `i`-th sampled subset. 

#### Registers
 - `ctrl`: An `(n - 1)`-qubit control register.
 - `junk [right]`: A right-sided junk register containing all intermediate masked copies, parity values, and ancilla qubits.
 - `target [right]`: The output bit of the approximate multi-controlled AND.


In [ ]:
from qualtran.bloqs.mcmt import ApproxMultiToffoli

### Example Instances

In [ ]:
sample_strings = (
    (1, 0),
    (0, 1),
)
approx_multi_toffoli = ApproxMultiToffoli(n=3, k=2, sample_strings=sample_strings)

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([approx_multi_toffoli],
           ['`approx_multi_toffoli`'])

### Call Graph

In [ ]:
from qualtran.resource_counting.generalizers import ignore_split_join
approx_multi_toffoli_g, approx_multi_toffoli_sigma = approx_multi_toffoli.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(approx_multi_toffoli_g)
show_counts_sigma(approx_multi_toffoli_sigma)